# optiver-close — Phase 1 on Colab

Reproduces the whole Phase 1 pipeline from nothing but the repo and a Kaggle account:
clone → download the competition data → build the fixture → **verify the rebuild against the
committed manifest** → run the 78-test suite → run the baseline and ablation scripts → read
the numbers.

The headline you are reproducing: pooled OOF MAE in bps over 300 purged, embargoed,
forward-chained validation dates —

| baseline | MAE (bps) | vs predict-zero |
|---|---|---|
| ridge (14 features) | 6.3224 | **−0.0628** |
| constant median | 6.3850 | −0.0002 |
| predict-zero (floor) | 6.3852 | — |
| carry, raw | 9.1023 | +2.7171 |

**Runtime:** CPU is fine — nothing here uses a GPU. Pick a **high-RAM** runtime if offered
(the baseline run peaks at ~4.5 GB RSS; the standard VM's ~12 GB is enough, but with less
margin). End to end: roughly 10–15 minutes, most of it the Kaggle download and the ablation
script.

**You need:** a Kaggle account that has **joined** the *Optiver — Trading at the Close*
competition (kaggle.com/competitions/optiver-trading-at-the-close → Late Submission /
"I Understand and Accept"). Credentials are handled by `kagglehub` — Colab secrets or an
interactive login prompt; no `kaggle.json` file needed. Without having accepted the rules,
the download in §2 returns a 403 no matter how you authenticate.

## 1. The repo

In [ ]:
# The repo is PRIVATE, so an anonymous clone will 404. Paste a GitHub personal access token
# with `repo` scope when prompted (read via getpass, never printed, lives only in this VM) —
# or make the repo public once and just press Enter:
#   gh repo edit Bromine185/optiver-close --visibility public
import os, subprocess, getpass
from pathlib import Path

REPO_USER = "Bromine185"
REPO_NAME = "optiver-close"
REPO = Path("/content") / REPO_NAME

if not REPO.exists():
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (blank if public): ")
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_USER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], check=True)
    del token, auth, url
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

%cd {REPO}
!git log --oneline -1

In [ ]:
# Colab ships numpy/pandas/pyarrow/scipy/sklearn/matplotlib. pytest is the usual gap.
import importlib, subprocess, sys

for module, package in [("pytest", "pytest>=8.0")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

sys.path.insert(0, str(REPO / "src"))
import numpy as np, pandas as pd
print(f"python {sys.version.split()[0]}  numpy {np.__version__}  pandas {pd.__version__}")

## 2. The raw data, from Kaggle

`data/raw/` is gitignored (641 MB) and `data/fixtures/train.parquet` (130 MB) is too — what
the repo commits is `manifest.json`, the record a rebuild is checked against, plus the 4 MB
smoke fixture the tests run on. So Colab downloads the competition data once and rebuilds.

Authentication is `kagglehub`'s problem, not this notebook's. It resolves credentials in
order — `KAGGLE_USERNAME`/`KAGGLE_KEY` env vars, a `~/.kaggle/kaggle.json` if one exists,
Colab secrets (key icon in the sidebar) — and when none are found, the cell falls back to
`kagglehub.login()`, an interactive prompt where you paste your username and API token
directly. No `kaggle.json` file is ever required.

In [ ]:
import json as _json
import shutil

RAW = REPO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

if not (RAW / "train.csv").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub"], check=True)
    import kagglehub

    try:
        src = Path(kagglehub.competition_download("optiver-trading-at-the-close"))
    except Exception as e:
        if "403" in str(e) or "Forbidden" in str(e):
            raise SystemExit(
                "403: this Kaggle account has not accepted the competition rules — "
                "visit kaggle.com/competitions/optiver-trading-at-the-close and join first."
            ) from e
        # no credentials found anywhere: interactive login (username + API token, no file)
        kagglehub.login()
        src = Path(kagglehub.competition_download("optiver-trading-at-the-close"))

    # kagglehub caches the files already unzipped; link them into data/raw/ where
    # scripts/build_fixture.py expects them (no 641 MB copy).
    for item in src.iterdir():
        dest = RAW / item.name
        if not dest.exists():
            dest.symlink_to(item)

print(f"train.csv: {(RAW / 'train.csv').stat().st_size / 1e6:.0f} MB")

## 3. Build the fixture, and check the rebuild against the committed manifest

`scripts/build_fixture.py` is the one place `train.csv` is ever read (CLAUDE.md
non-negotiable #2). It downcasts with a measured precision gate, refuses to fill nulls, and
writes `manifest.json` — row counts, dtypes, per-column null counts, coverage, the gate's
numbers, and the parquet's sha256.

The build **overwrites** the committed manifest, which is exactly what makes verification
easy: `git diff` afterwards is the reproduction report. Two classes of difference are
expected and harmless — timestamps/`build_seconds`/`versions`, and the parquet `sha256`/
`bytes` if Colab's pandas/pyarrow differ from the versions recorded (zstd bytes are
version-sensitive; the *content* checks are everything else). **Any other diff — a row
count, a null count, a coverage number, the precision gate — means the rebuild did not
reproduce the fixture, and nothing downstream of it should be trusted.**

In [ ]:
!python scripts/build_fixture.py

In [ ]:
# The reproduction report. Silence outside the known-noise keys is success.
diff = subprocess.run(
    ["git", "diff", "--unified=0", "--", "data/fixtures/manifest.json"],
    capture_output=True, text=True, check=True,
).stdout

NOISE = ("built_at_utc", "build_seconds", "mtime_utc", "sha256", "bytes",
         "pandas", "numpy", "python", "smoke_rebuilt_at_utc", "@@", "---", "+++",
         "diff --git", "index ")
real = [l for l in diff.splitlines()
        if l[:1] in "+-" and not any(k in l for k in NOISE)]
if real:
    print("MANIFEST MISMATCH — the rebuild changed recorded content:")
    print("\n".join(real))
    raise SystemExit("do not trust anything below this cell")
print("manifest content reproduced (only timestamps/versions/compression bytes moved)")

man = _json.loads((REPO / "data" / "fixtures" / "manifest.json").read_text())
print(f"rows {man['rows']:,}  build {man['build_seconds']}s")

## 4. The test suite

78 tests. They pin the split guarantees (no date ever divided, embargo respected, forward
chaining strict — including on a date axis with holes), the target's definition (the
60-second horizon, the shared index leg), the null-handling policy, and determinism. With
the full fixture present, all 78 run; on a bare clone two would skip.

In [ ]:
!python -m pytest -q

## 5. Baselines — the honest floor and what beats it

`run_baselines.py` on the FULL preset: 5 folds × 60 validation dates, 5-date embargo,
expanding window. Peak ~4.5 GB RSS. Every number is a difference against predict-zero,
because the target is index-relative and near-zero-mean *by construction* — 6.4 bps MAE is
not a straw man, it is within 0.0002 bps of the best constant that exists.

In [ ]:
!python scripts/run_baselines.py --preset FULL

In [ ]:
rep = _json.loads((REPO / "reports" / "phase1_baselines.json").read_text())

print(f"{'model':<18s} {'MAE bps':>9s} {'vs zero':>9s} {'%':>7s}")
for row in sorted(rep["scorecard"], key=lambda r: r["mae_bps"]):
    print(f"{row['model']:<18s} {row['mae_bps']:9.4f} {-row['vs_zero_bps']:+9.4f} "
          f"{row['vs_zero_pct']:6.2f}%")
print(f"\nbest_model = {rep['best_model']}   runtime {rep['runtime_seconds']:.0f}s")

coef = pd.Series(rep["ridge_coefficients_mean"]).sort_values(key=abs, ascending=False)
print("\ntop ridge coefficients (mean over folds, standardized features):")
print(coef.head(6).round(3).to_string())

## 6. Ablations

The slowest script: feature ablations for the ridge, the carry baseline's autocorrelation
analysis, and the per-stock spread. Skip this cell if you only wanted the floor and the
headline.

In [ ]:
!python scripts/run_ablations.py --preset FULL

## 7. Read the reports

Both scripts write machine-readable JSON under `reports/` — the same files whose committed
copies back every number in `RESEARCH.md`. Colab's copies should agree with the committed
ones to reporting precision; fold-to-fold MAE genuinely varies by 1.3 bps (that is the
data, not noise — see "Known limitations" in CLAUDE.md), so compare pooled numbers, not
single folds.

In [ ]:
for name in ("phase1_baselines.json", "phase1_ablations.json"):
    p = REPO / "reports" / name
    if not p.exists():
        print(f"{name}: not written (section skipped?)")
        continue
    d = subprocess.run(["git", "diff", "--stat", "--", str(p.relative_to(REPO))],
                       capture_output=True, text=True).stdout.strip()
    print(f"{name}: {'matches the committed copy' if not d else d}")

In [ ]:
# Per-bucket picture: the ridge's edge is ~5% at the open of the auction and decays to ~1%.
import matplotlib.pyplot as plt

bs = pd.DataFrame(rep["by_seconds"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(bs["seconds_in_bucket"], bs["mae_zero"], ls="--", label="predict-zero")
axes[0].plot(bs["seconds_in_bucket"], bs["mae"], label="best model")
axes[0].set_ylabel("MAE (bps)"); axes[0].legend()
axes[1].plot(bs["seconds_in_bucket"], bs["improvement_pct"])
axes[1].set_ylabel("improvement vs zero (%)")
for ax in axes:
    ax.set_xlabel("seconds_in_bucket")
fig.suptitle("MAE through the auction")
plt.tight_layout(); plt.show()

---

### What NOT to conclude from a green run

Phase 1 is a harness and a floor. A successful reproduction here says the split is
trustworthy and the floor is honest — it says nothing about what a real model will do.
The ridge's −0.06 bps is 1% better than doing nothing; Phase 2 (LightGBM on rolling
features, optimising MAE directly) is where a model claim would come from, and it will be
scored by this same harness so the numbers stay comparable.

Nothing in this notebook needs to leave the VM — the fixture and reports are rebuilt from
the manifest anywhere. If you want to keep Colab's reports anyway:

```python
from google.colab import files
files.download("reports/phase1_baselines.json")
```